# ИИ-ассистент режиссёра — учебный прототип

Этот ноутбук реализует полный базовый конвейер из кейса **«ИИ-ассистент режиссёра»**.

Основная идея:

**художественный текст → сцены → персонажи → факты и цитаты → цели/интерпретации → сценические решения → сохранённый проект → вопросы по проекту**

Ноутбук специально написан простым Python-кодом, чтобы каждую часть можно было прочитать и понять школьнику.

## Что закрывает этот ноутбук

- принимает текст пьесы из `.txt`;
- делит пьесу на сцены;
- учитывает говорящих персонажей и персонажей, которые появляются только в ремарках;
- создаёт карточку **каждой** сцены;
- хранит факты отдельно от интерпретаций;
- прикладывает цитату к каждому фактическому событию и к гипотезе о цели;
- строит линию персонажа через все сцены;
- предлагает несколько сценических вариантов **для каждой сцены**;
- учитывает ограничения постановки;
- позволяет режиссёру менять название, событие, цель и заметку;
- сохраняет изменения в JSON и умеет загрузить их обратно;
- отвечает на вопросы только по текущему сохранённому проекту;
- содержит автоматические проверки по критериям задания.

Важно: в исходном репозитории нет готового разобранного проекта и готовой пьесы, поэтому в ноутбуке используется маленький демонстрационный текст. Для реальной проверки нужно указать путь к `.txt`-пьесе с открытой лицензией.

## 1. Почему решение сделано именно так

`README.md` и ТЗ предупреждают, что просто передать всю пьесу одному чат-боту недостаточно: модель может перепутать события, придумать детали и потерять состояние проекта.

Поэтому здесь используется **узкий конвейер**:

1. сначала структурируем текст;
2. затем работаем с отдельными сценами;
3. храним цитаты, факты и интерпретации отдельно;
4. сценическое решение строится только из персонажей текущей сцены;
5. правки режиссёра записываются в проект;
6. вопросы задаются уже проекту, а не новому пустому чату.

Для учебной версии не нужна большая языковая модель. Правила и небольшие функции проще проверять и объяснять. Это соответствует требованию исследовать компактный подход и узкий конвейер.

## 2. Импорты

In [1]:
import json
import re
from pathlib import Path
from copy import deepcopy
from tempfile import TemporaryDirectory

## 3. Загрузка пьесы

Используем обычный UTF-8 `.txt`.

- `TEXT_PATH = ""` — запустить на демонстрационном тексте.
- `TEXT_PATH = "chaika.txt"` — использовать свою пьесу.

В репозиторий кейса готовые тексты пьес класть не нужно.

In [2]:
def load_play_text(path):
    """Читаем пьесу из UTF-8 текстового файла."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path}")
    return path.read_text(encoding="utf-8")


DEMO_TEXT = """
ДЕЙСТВИЕ ПЕРВОЕ

ЯВЛЕНИЕ ПЕРВОЕ
(Сад. Входит Маша.)
Маша.
Мне нужно поговорить с Петром.
Пётр.
Я пришёл узнать, что случилось.
Маша.
Останься. Нам нельзя расходиться.

ЯВЛЕНИЕ ВТОРОЕ
(Входит Ольга. Маша отходит к окну.)
Ольга.
Я слышала ваш разговор.
Пётр.
Не вмешивайся, Ольга.
Ольга.
Я хочу помочь вам.

ДЕЙСТВИЕ ВТОРОЕ

ЯВЛЕНИЕ ПЕРВОЕ
(Пётр возвращается. Все молчат.)
Пётр.
Теперь нужно принять решение.
Маша.
Я не хочу уходить.
Ольга.
Давайте начнём сначала.
"""

TEXT_PATH = ""
play_text = load_play_text(TEXT_PATH) if TEXT_PATH else DEMO_TEXT
print(play_text[:1200])


ДЕЙСТВИЕ ПЕРВОЕ

ЯВЛЕНИЕ ПЕРВОЕ
(Сад. Входит Маша.)
Маша.
Мне нужно поговорить с Петром.
Пётр.
Я пришёл узнать, что случилось.
Маша.
Останься. Нам нельзя расходиться.

ЯВЛЕНИЕ ВТОРОЕ
(Входит Ольга. Маша отходит к окну.)
Ольга.
Я слышала ваш разговор.
Пётр.
Не вмешивайся, Ольга.
Ольга.
Я хочу помочь вам.

ДЕЙСТВИЕ ВТОРОЕ

ЯВЛЕНИЕ ПЕРВОЕ
(Пётр возвращается. Все молчат.)
Пётр.
Теперь нужно принять решение.
Маша.
Я не хочу уходить.
Ольга.
Давайте начнём сначала.



## 4. Предварительная обработка текста

In [3]:
def normalize_text(text):
    """Убираем лишние пробелы и нормализуем переносы строк."""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [line.strip() for line in text.split("\n")]

    result = []
    previous_empty = False

    for line in lines:
        if line == "":
            if not previous_empty:
                result.append("")
            previous_empty = True
        else:
            result.append(line)
            previous_empty = False

    return "\n".join(result).strip()


play_text = normalize_text(play_text)
print("Символов:", len(play_text))
print("Строк:", len(play_text.splitlines()))

Символов: 462
Строк: 30


## 5. Разделение текста на сцены

Сначала ищем подробные заголовки (`ЯВЛЕНИЕ`, `СЦЕНА`, `SCENE`).

Если их нет, используем `ДЕЙСТВИЕ` / `ACT`.

Название сцены сохраняет и номер действия. Например:

`ДЕЙСТВИЕ ПЕРВОЕ — ЯВЛЕНИЕ ПЕРВОЕ`

Так режиссёр не теряет контекст исходного текста.

In [4]:
DETAIL_SCENE_RE = re.compile(r"^(ЯВЛЕНИЕ|СЦЕНА|SCENE)\b.*$", re.IGNORECASE)
ACT_RE = re.compile(r"^(ДЕЙСТВИЕ|АКТ|ACT)\b.*$", re.IGNORECASE)


def split_into_scenes(text):
    lines = text.splitlines()
    detailed_indexes = [i for i, line in enumerate(lines) if DETAIL_SCENE_RE.match(line)]

    if detailed_indexes:
        starts = detailed_indexes
    else:
        starts = [i for i, line in enumerate(lines) if ACT_RE.match(line)]

    scenes = []

    if not starts:
        blocks = []
        current = []
        for line in lines:
            if line == "" and current:
                blocks.append(current)
                current = []
            elif line != "":
                current.append(line)
        if current:
            blocks.append(current)

        return [
            {"id": i + 1, "title": f"Сцена {i + 1}", "text": "\n".join(block)}
            for i, block in enumerate(blocks)
        ]

    current_act = ""

    act_indexes = [i for i, line in enumerate(lines) if ACT_RE.match(line)]

    for pos, start in enumerate(starts):
        end = starts[pos + 1] if pos + 1 < len(starts) else len(lines)
        marker = lines[start].strip()

        # Для подробной сцены ищем ближайшее предыдущее действие.
        previous_acts = [i for i in act_indexes if i < start]
        if previous_acts:
            current_act = lines[previous_acts[-1]].strip()

        if ACT_RE.match(marker):
            current_act = marker
            title = marker
        else:
            title = f"{current_act} — {marker}" if current_act else marker

        block_lines = [line for line in lines[start:end] if line != ""]
        scenes.append({
            "id": len(scenes) + 1,
            "title": title,
            "text": "\n".join(block_lines)
        })

    return scenes


raw_scenes = split_into_scenes(play_text)
print("Найдено сцен:", len(raw_scenes))
for scene in raw_scenes:
    print(scene["id"], scene["title"])

Найдено сцен: 3
1 ДЕЙСТВИЕ ПЕРВОЕ — ЯВЛЕНИЕ ПЕРВОЕ
2 ДЕЙСТВИЕ ПЕРВОЕ — ЯВЛЕНИЕ ВТОРОЕ
3 ДЕЙСТВИЕ ВТОРОЕ — ЯВЛЕНИЕ ПЕРВОЕ


## 6. Поиск говорящих персонажей

В разных пьесах имя персонажа может выглядеть как `Маша.` или как `МАША`.

Поэтому поддерживаем два простых варианта. Дополнительно отбрасываем известные слова сценографии, чтобы не принять `Сад.` за персонажа.

In [5]:
STAGE_WORDS = {
    "сад", "комната", "улица", "дом", "тишина", "занавес", "свет",
    "сцена", "вечер", "утро", "ночь", "все", "тишина"
}

SPEAKER_WITH_DOT_RE = re.compile(
    r"^[А-ЯЁA-Z][А-ЯЁA-Zа-яёa-z0-9_-]*(?:\s+[А-ЯЁA-Z][А-ЯЁA-Zа-яёa-z0-9_-]*){0,4}\.$"
)
SPEAKER_UPPER_RE = re.compile(r"^[А-ЯЁA-Z][А-ЯЁA-Z0-9 _-]{1,40}$")


def is_stage_direction(line):
    line = line.strip()
    return line.startswith("(") or line.startswith("[") or line.startswith("{")


def looks_like_speaker(lines, index):
    line = lines[index].strip()

    match = SPEAKER_WITH_DOT_RE.match(line) or SPEAKER_UPPER_RE.match(line)
    if not match:
        return False

    name = line.rstrip(".").strip()
    if name.lower() in STAGE_WORDS:
        return False

    # Заголовок сцены не может быть именем персонажа.
    if DETAIL_SCENE_RE.match(line) or ACT_RE.match(line):
        return False

    # После имени должна идти реплика или внутренняя ремарка, а не новый заголовок.
    if index + 1 < len(lines):
        next_line = lines[index + 1].strip()
        if not next_line:
            return False
        if DETAIL_SCENE_RE.match(next_line) or ACT_RE.match(next_line):
            return False

    return True


def normalize_name(name):
    return re.sub(r"\s+", " ", name.strip(" .,:;!?"))

## 7. Персонажи, которые появляются только в ремарке

Критерии отдельно проверяют ситуацию, когда персонаж не произносит ни слова, но входит на сцену.

Поэтому ищем простые конструкции:

- `входит Маша`;
- `выходит Пётр`;
- `появляется Ольга`;
- `возвращается Маша и Пётр`.

Это не полноценный лингвистический анализ, а прозрачное правило для учебного прототипа.

In [6]:
ENTRY_RE = re.compile(
    r"\b(входит|входят|выходит|выходят|появляется|появляются|приходит|приходят|"
    r"возвращается|возвращаются)\s+([^.,;()]+)",
    re.IGNORECASE
)


def extract_entry_names(line):
    names = []
    for match in ENTRY_RE.finditer(line):
        part = match.group(2).strip()
        pieces = re.split(r"\s+и\s+|,\s*", part)
        for piece in pieces:
            piece = normalize_name(piece)
            if piece and piece.lower() not in STAGE_WORDS:
                names.append(piece)
    return names


def extract_scene_characters(scene_text):
    lines = scene_text.splitlines()
    participants = []
    speakers = {}
    directions = []
    current_speaker = None

    for i, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue

        if DETAIL_SCENE_RE.match(line) or ACT_RE.match(line):
            current_speaker = None
            continue

        if is_stage_direction(line):
            directions.append(line)
            for name in extract_entry_names(line):
                if name not in participants:
                    participants.append(name)
            current_speaker = None
            continue

        if looks_like_speaker(lines, i):
            name = normalize_name(line)
            if name not in participants:
                participants.append(name)
            speakers.setdefault(name, 0)
            current_speaker = name
            continue

        if current_speaker:
            speakers[current_speaker] += 1

    return participants, speakers, directions

## 8. Сбор реплик

In [7]:
def collect_speeches(scene_text):
    """Возвращает реплики в виде {персонаж: [цитата, ...]}."""
    lines = scene_text.splitlines()
    speeches = {}
    current_speaker = None
    current_parts = []

    def save_current():
        nonlocal current_speaker, current_parts
        if current_speaker and current_parts:
            quote = " ".join(current_parts).strip()
            if quote:
                speeches.setdefault(current_speaker, []).append(quote)
        current_speaker = None
        current_parts = []

    for i, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue

        if DETAIL_SCENE_RE.match(line) or ACT_RE.match(line):
            save_current()
            continue

        if looks_like_speaker(lines, i):
            save_current()
            current_speaker = normalize_name(line)
            continue

        if is_stage_direction(line):
            continue

        if current_speaker:
            current_parts.append(line)

    save_current()
    return speeches

## 9. Цитаты, факты и интерпретации

Это принципиальное разделение из задания.

### Факт
Его можно проверить непосредственно по исходному тексту. Например:

`(Сад. Входит Маша.)`

### Интерпретация
Это предположение режиссёрского разбора. Например:

`Маша хочет удержать Петра рядом.`

Интерпретация всегда получает тип `interpretation`, а рядом хранится цитата, на которой она основана.

In [8]:
GOAL_RULES = [
    (r"\bостанься\b", "убедить другого человека остаться"),
    (r"\bуйди\b|\bуходи\b", "добиться ухода собеседника"),
    (r"\bпомоги\b|\bпомочь\b", "получить помощь"),
    (r"\bхочу\b", "добиться того, чего герой хочет"),
    (r"\bнужно\b|\bнадо\b", "добиться выполнения необходимого действия"),
    (r"\bрешение\b", "добиться принятия решения"),
    (r"\bначнём\b|\bначнем\b", "начать заново или изменить ситуацию"),
    (r"\bдавай\b", "предложить совместное действие"),
]


def make_goal_hypothesis(speaker, quote):
    """Очень простая гипотеза о цели. Это НЕ факт текста."""
    for pattern, goal in GOAL_RULES:
        if re.search(pattern, quote, re.IGNORECASE):
            return {
                "type": "interpretation",
                "text": f"{speaker} может стремиться: {goal}.",
                "quote": quote
            }

    return {
        "type": "interpretation",
        "text": f"Возможная цель {speaker}: продолжить разговор и разобраться в ситуации.",
        "quote": quote
    }


def first_useful_quote(speeches, directions, scene_text):
    for values in speeches.values():
        if values:
            return values[0]
    for direction in directions:
        return direction
    lines = [line for line in scene_text.splitlines() if line and not DETAIL_SCENE_RE.match(line) and not ACT_RE.match(line)]
    return lines[0] if lines else ""


def build_events(speeches, directions):
    events = []

    # 1. Ремарки — самые надёжные факты действия на сцене.
    for direction in directions:
        events.append({
            "type": "fact",
            "text": direction,
            "quote": direction
        })

    # 2. Реплики с явным действием тоже можно использовать как факт речи.
    action_words = [r"останься", r"уйди", r"помоги", r"нужно", r"надо", r"хочу", r"решение", r"начн"]
    action_re = re.compile("|".join(action_words), re.IGNORECASE)

    for speaker, quotes in speeches.items():
        for quote in quotes:
            if action_re.search(quote):
                events.append({
                    "type": "fact",
                    "text": f"{speaker} произносит реплику: {quote}",
                    "quote": quote
                })

    # Даже если ремарок и явных слов действия нет, событие должно существовать.
    if not events:
        for speaker, quotes in speeches.items():
            if quotes:
                events.append({
                    "type": "fact",
                    "text": f"{speaker} говорит: {quotes[0]}",
                    "quote": quotes[0]
                })
                break

    return events[:8]

## 10. Карточка одной сцены

In [9]:
def build_scene_card(scene):
    participants, speaker_counts, directions = extract_scene_characters(scene["text"])
    speeches = collect_speeches(scene["text"])
    events = build_events(speeches, directions)

    goals = {}
    for speaker, quotes in speeches.items():
        if quotes:
            goals[speaker] = make_goal_hypothesis(speaker, quotes[0])
        else:
            goals[speaker] = {
                "type": "interpretation",
                "text": f"Цель {speaker} пока не определена по репликам этой сцены.",
                "quote": ""
            }

    # Если персонаж молчит, его цель не выдумываем.
    for participant in participants:
        goals.setdefault(participant, {
            "type": "interpretation",
            "text": f"Для {participant} в этой сцене нет собственной реплики; цель не определена.",
            "quote": ""
        })

    return {
        "id": scene["id"],
        "title": scene["title"],
        "source_text": scene["text"],
        "participants": participants,
        "speaker_counts": speaker_counts,
        "speeches": speeches,
        "stage_directions": directions,
        "events": events,
        "goals": goals,
        "director_note": "",
        "staging_options": []
    }


scene_cards = [build_scene_card(scene) for scene in raw_scenes]
print("Карточки созданы:", len(scene_cards))
print(json.dumps(scene_cards[0], ensure_ascii=False, indent=2)[:4000])

Карточки созданы: 3
{
  "id": 1,
  "title": "ДЕЙСТВИЕ ПЕРВОЕ — ЯВЛЕНИЕ ПЕРВОЕ",
  "source_text": "ЯВЛЕНИЕ ПЕРВОЕ\n(Сад. Входит Маша.)\nМаша.\nМне нужно поговорить с Петром.\nПётр.\nЯ пришёл узнать, что случилось.\nМаша.\nОстанься. Нам нельзя расходиться.",
  "participants": [
    "Маша",
    "Пётр"
  ],
  "speaker_counts": {
    "Маша": 2,
    "Пётр": 1
  },
  "speeches": {
    "Маша": [
      "Мне нужно поговорить с Петром.",
      "Останься. Нам нельзя расходиться."
    ],
    "Пётр": [
      "Я пришёл узнать, что случилось."
    ]
  },
  "stage_directions": [
    "(Сад. Входит Маша.)"
  ],
  "events": [
    {
      "type": "fact",
      "text": "(Сад. Входит Маша.)",
      "quote": "(Сад. Входит Маша.)"
    },
    {
      "type": "fact",
      "text": "Маша произносит реплику: Мне нужно поговорить с Петром.",
      "quote": "Мне нужно поговорить с Петром."
    },
    {
      "type": "fact",
      "text": "Маша произносит реплику: Останься. Нам нельзя расходиться.",
      "quote": "О

## 11. Линия персонажа через всю пьесу

In [10]:
def build_character_lines(scene_cards):
    lines = {}

    for scene in scene_cards:
        for character in scene["participants"]:
            lines.setdefault(character, []).append({
                "scene_id": scene["id"],
                "scene_title": scene["title"],
                "participation": "говорит" if character in scene["speeches"] else "есть в сцене, но не говорит",
                "goal": scene["goals"][character]["text"]
            })

    return lines

## 12. Ограничения постановки

Ограничения храним отдельно от текста пьесы. Это позволяет учитывать реальные условия постановки и менять их без изменения исходника.

In [11]:
def default_constraints():
    return {
        "space": "малая сцена",
        "lighting": "минимум света",
        "actors_limit": 5,
        "style": "литературный и пластический варианты",
        "props": "минимум реквизита"
    }


def set_constraints(project, **changes):
    for key, value in changes.items():
        if key in project["constraints"]:
            project["constraints"][key] = value

    project["history"].append({
        "action": "set_constraints",
        "changes": changes
    })

## 13. Сценические решения для каждой сцены

Минимальный результат требует хотя бы одного решения. Для сильного варианта полезно иметь альтернативы.

Поэтому здесь для **каждой** сцены создаются три простых варианта:

1. литературный;
2. пластический;
3. минимальное пространство.

Важно: персонажи в решении берутся только из карточки этой сцены. Это защищает от выдуманных героев.

In [12]:
def get_scene_quote(scene):
    for event in scene["events"]:
        if event["quote"]:
            return event["quote"]
    return first_useful_quote(scene["speeches"], scene["stage_directions"], scene["source_text"])


def make_staging_option(scene, project, style):
    characters = scene["participants"]
    quote = get_scene_quote(scene)
    names = ", ".join(characters) if characters else "персонажи не определены"
    limits = project["constraints"]

    if style == "литературный":
        steps = [
            f"Расположить {names} так, чтобы зрителю было легко следить за диалогом.",
            "Сохранить текст реплик без сокращения.",
            f"Использовать ограничение пространства: {limits['space']}.",
            f"Опора на текст: «{quote}»"
        ]
    elif style == "пластический":
        steps = [
            f"Передать отношения между {names} прежде всего через расстояние и движение.",
            "Сократить количество слов, но оставить смысл ключевой реплики.",
            f"Свет использовать экономно: {limits['lighting']}.",
            f"Опора на текст: «{quote}»"
        ]
    else:
        steps = [
            f"Оставить на площадке только {limits['props']}.",
            f"Построить сцену вокруг действий {names}.",
            f"Не превышать лимит актёров: {limits['actors_limit']}.",
            f"Опора на текст: «{quote}»"
        ]

    return {
        "style": style,
        "characters_used": list(characters),
        "based_on_quote": quote,
        "steps": steps,
        "interpretation": "Это режиссёрское предложение, а не факт исходного текста."
    }


def build_all_staging_options(project):
    for scene in project["scenes"]:
        scene["staging_options"] = [
            make_staging_option(scene, project, style)
            for style in ["литературный", "пластический", "минимальное пространство"]
        ]

## 14. Создание проекта

In [13]:
def create_project(play_text, scenes):
    project = {
        "project_name": "ИИ-ассистент режиссёра",
        "source_length": len(play_text),
        "scenes": deepcopy(scenes),
        "character_lines": {},
        "constraints": default_constraints(),
        "history": []
    }

    project["character_lines"] = build_character_lines(project["scenes"])
    build_all_staging_options(project)
    return project


project = create_project(play_text, scene_cards)

print("Проект создан.")
print("Сцен:", len(project["scenes"]))
print("Персонажей:", len(project["character_lines"]))
print("Ограничения:", project["constraints"])

Проект создан.
Сцен: 3
Персонажей: 3
Ограничения: {'space': 'малая сцена', 'lighting': 'минимум света', 'actors_limit': 5, 'style': 'литературный и пластический варианты', 'props': 'минимум реквизита'}


## 15. Просмотр одной сцены

In [14]:
def show_scene(project, scene_id):
    """Печатает понятную карточку одной сцены."""
    scene = next((s for s in project["scenes"] if s["id"] == scene_id), None)
    if scene is None:
        raise ValueError("Сцена не найдена")

    print(f"=== СЦЕНА {scene['id']}: {scene['title']} ===")
    print("\nУчастники:", ", ".join(scene["participants"]) or "не найдены")

    print("\nФакты и события:")
    for event in scene["events"]:
        print(f"- {event['text']}")
        print(f"  Цитата: «{event['quote']}»")

    print("\nЦели / интерпретации:")
    for name, goal in scene["goals"].items():
        print(f"- {name}: {goal['text']}")
        if goal["quote"]:
            print(f"  Основание: «{goal['quote']}»")

    print("\nЗаметка режиссёра:", scene["director_note"] or "нет")

    print("\nСценические варианты:")
    for option in scene["staging_options"]:
        print(f"\n[{option['style']}]")
        print("Персонажи:", ", ".join(option["characters_used"]) or "нет")
        print("Цитата:", option["based_on_quote"])
        for step in option["steps"]:
            print("-", step)


show_scene(project, 1)

=== СЦЕНА 1: ДЕЙСТВИЕ ПЕРВОЕ — ЯВЛЕНИЕ ПЕРВОЕ ===

Участники: Маша, Пётр

Факты и события:
- (Сад. Входит Маша.)
  Цитата: «(Сад. Входит Маша.)»
- Маша произносит реплику: Мне нужно поговорить с Петром.
  Цитата: «Мне нужно поговорить с Петром.»
- Маша произносит реплику: Останься. Нам нельзя расходиться.
  Цитата: «Останься. Нам нельзя расходиться.»

Цели / интерпретации:
- Маша: Маша может стремиться: добиться выполнения необходимого действия.
  Основание: «Мне нужно поговорить с Петром.»
- Пётр: Возможная цель Пётр: продолжить разговор и разобраться в ситуации.
  Основание: «Я пришёл узнать, что случилось.»

Заметка режиссёра: нет

Сценические варианты:

[литературный]
Персонажи: Маша, Пётр
Цитата: (Сад. Входит Маша.)
- Расположить Маша, Пётр так, чтобы зрителю было легко следить за диалогом.
- Сохранить текст реплик без сокращения.
- Использовать ограничение пространства: малая сцена.
- Опора на текст: «(Сад. Входит Маша.)»

[пластический]
Персонажи: Маша, Пётр
Цитата: (Сад. Входит

## 16. Правки режиссёра

In [15]:
def get_scene(project, scene_id):
    scene = next((s for s in project["scenes"] if s["id"] == scene_id), None)
    if scene is None:
        raise ValueError(f"Сцена {scene_id} не найдена")
    return scene


def rename_scene(project, scene_id, new_title):
    scene = get_scene(project, scene_id)
    old_title = scene["title"]
    scene["title"] = new_title

    for items in project["character_lines"].values():
        for item in items:
            if item["scene_id"] == scene_id:
                item["scene_title"] = new_title

    project["history"].append({
        "action": "rename_scene",
        "scene_id": scene_id,
        "old": old_title,
        "new": new_title
    })


def edit_event(project, scene_id, event_number, new_text, new_quote):
    """Исправляем факт только вместе с новой цитатой из исходного текста."""
    scene = get_scene(project, scene_id)
    index = event_number - 1
    if not (0 <= index < len(scene["events"])):
        raise IndexError("Нет такого события")

    if not new_quote or new_quote not in scene["source_text"]:
        raise ValueError("Для факта нужна точная цитата из исходного текста сцены.")

    event = scene["events"][index]
    old_text = event["text"]
    old_quote = event["quote"]
    event["text"] = new_text
    event["quote"] = new_quote
    event["type"] = "fact"

    project["history"].append({
        "action": "edit_event",
        "scene_id": scene_id,
        "event_number": event_number,
        "old": old_text,
        "new": new_text,
        "old_quote": old_quote,
        "new_quote": new_quote
    })


def edit_goal(project, scene_id, character, new_text):
    scene = get_scene(project, scene_id)
    if character not in scene["goals"]:
        raise ValueError("Такого персонажа нет в сцене")

    old_text = scene["goals"][character]["text"]
    scene["goals"][character]["text"] = new_text
    scene["goals"][character]["type"] = "interpretation"

    project["history"].append({
        "action": "edit_goal",
        "scene_id": scene_id,
        "character": character,
        "old": old_text,
        "new": new_text
    })


def set_director_note(project, scene_id, note):
    scene = get_scene(project, scene_id)
    old_note = scene["director_note"]
    scene["director_note"] = note

    project["history"].append({
        "action": "director_note",
        "scene_id": scene_id,
        "old": old_note,
        "new": note
    })


def refresh_staging_for_scene(project, scene_id):
    scene = get_scene(project, scene_id)
    scene["staging_options"] = [
        make_staging_option(scene, project, style)
        for style in ["литературный", "пластический", "минимальное пространство"]
    ]

## 17. Пример реальной режиссёрской правки

Изменим название, добавим заметку и поменяем одну интерпретацию.

Главное: правки идут **в объект проекта**, поэтому следующий вопрос получает уже обновлённое состояние.

In [16]:
rename_scene(project, 1, "Встреча после разговора")
set_director_note(project, 1, "Между Машей и Петром держать заметную дистанцию до ключевой реплики.")
edit_goal(project, 1, "Маша", "Маша пытается убедить Петра остаться рядом.")
refresh_staging_for_scene(project, 1)

show_scene(project, 1)

=== СЦЕНА 1: Встреча после разговора ===

Участники: Маша, Пётр

Факты и события:
- (Сад. Входит Маша.)
  Цитата: «(Сад. Входит Маша.)»
- Маша произносит реплику: Мне нужно поговорить с Петром.
  Цитата: «Мне нужно поговорить с Петром.»
- Маша произносит реплику: Останься. Нам нельзя расходиться.
  Цитата: «Останься. Нам нельзя расходиться.»

Цели / интерпретации:
- Маша: Маша пытается убедить Петра остаться рядом.
  Основание: «Мне нужно поговорить с Петром.»
- Пётр: Возможная цель Пётр: продолжить разговор и разобраться в ситуации.
  Основание: «Я пришёл узнать, что случилось.»

Заметка режиссёра: Между Машей и Петром держать заметную дистанцию до ключевой реплики.

Сценические варианты:

[литературный]
Персонажи: Маша, Пётр
Цитата: (Сад. Входит Маша.)
- Расположить Маша, Пётр так, чтобы зрителю было легко следить за диалогом.
- Сохранить текст реплик без сокращения.
- Использовать ограничение пространства: малая сцена.
- Опора на текст: «(Сад. Входит Маша.)»

[пластический]
Персонаж

## 18. Сохранение и загрузка проекта

In [17]:
def save_project(project, path="director_project.json"):
    """Сохраняем именно текущее состояние проекта."""
    path = Path(path)
    path.write_text(
        json.dumps(project, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    return path


def load_project(path="director_project.json"):
    """Загружаем ранее сохранённое состояние."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Проект не найден: {path}")
    return json.loads(path.read_text(encoding="utf-8"))

## 19. Поиск по сохранённому проекту

Это учебная версия идеи RAG: сначала ищем подходящие сцены среди **сохранённых данных проекта**, а уже потом формируем ответ.

Никаких данных «из головы» в ответ не добавляем.

In [18]:
STOP_WORDS = {
    "и", "в", "во", "на", "по", "к", "с", "со", "из", "за", "что", "это", "как",
    "у", "а", "о", "об", "про", "мне", "моя", "мой", "какой", "какая", "где", "кто",
    "есть", "эта", "этой", "эти", "этот"
}


def words(text):
    return {
        w.lower()
        for w in re.findall(r"[А-ЯЁA-Zа-яёa-z0-9-]+", text)
        if w.lower() not in STOP_WORDS
    }


def scene_search_text(scene):
    parts = [
        scene["title"],
        " ".join(scene["participants"]),
        scene["director_note"],
        scene["source_text"]
    ]
    for event in scene["events"]:
        parts.extend([event["text"], event["quote"]])
    for goal in scene["goals"].values():
        parts.extend([goal["text"], goal["quote"]])
    for option in scene["staging_options"]:
        parts.extend([option["style"], option["based_on_quote"]])
    return " ".join(parts)


def search_project(project, query, limit=3):
    query_words = words(query)
    scored = []

    for scene in project["scenes"]:
        score = len(query_words & words(scene_search_text(scene)))
        if score > 0:
            scored.append((score, scene))

    scored.sort(key=lambda item: item[0], reverse=True)
    return [scene for _, scene in scored[:limit]]

## 20. Ответы на вопросы по текущему проекту

In [19]:
def answer_about_scene(scene, query):
    q = query.lower()

    if "как называется" in q or "название" in q:
        return f"Текущее название сцены: «{scene['title']}»."

    if "кто" in q and ("участв" in q or "персонаж" in q or "люд" in q):
        names = ", ".join(scene["participants"]) or "не определены"
        return f"В сцене участвуют: {names}."

    if "событ" in q:
        facts = [event["text"] for event in scene["events"]]
        return "Факты текста:\n- " + "\n- ".join(facts) if facts else "В проекте события не найдены."

    if "цитат" in q:
        quotes = [event["quote"] for event in scene["events"] if event["quote"]]
        return "Цитаты:\n- " + "\n- ".join(quotes) if quotes else "В проекте цитаты не найдены."

    if "цель" in q or "мотивац" in q:
        parts = []
        for name, goal in scene["goals"].items():
            if goal["quote"]:
                parts.append(f"{name}: {goal['text']} Основание: «{goal['quote']}»")
            else:
                parts.append(f"{name}: {goal['text']}")
        return "Интерпретации целей:\n- " + "\n- ".join(parts)

    if "решен" in q or "мизансцен" in q or "постанов" in q:
        return "Варианты:\n" + "\n".join(
            f"- {o['style']}: {o['steps'][0]}"
            for o in scene["staging_options"]
        )

    if "замет" in q or "правк" in q:
        return "Заметка режиссёра: " + (scene["director_note"] or "нет")

    return ""


def ask_project(project, query):
    """Отвечаем только по текущему проекту."""
    q = query.lower()

    # Явные ссылки на сцену №1 не должны зависеть от её названия.
    if "первая сцена" in q and project["scenes"]:
        found = [project["scenes"][0]]
    elif "сцена 1" in q and project["scenes"]:
        found = [project["scenes"][0]]
    elif "замет" in q and any(scene["director_note"] for scene in project["scenes"]):
        found = [scene for scene in project["scenes"] if scene["director_note"]]
    else:
        found = search_project(project, query)

    if not found:
        return "В сохранённом проекте не найдено подтверждения для этого вопроса."

    answers = []
    for scene in found:
        answer = answer_about_scene(scene, query)
        if answer:
            answers.append(f"Сцена «{scene['title']}»:\n{answer}")

    if not answers:
        # Общий безопасный ответ без выдумывания новых фактов.
        return "В найденных сценах нет готового ответа на этот вопрос."

    return "\n\n".join(answers)

## 21. Проверяем память проекта

In [20]:
print(ask_project(project, "Как называется первая сцена?"))

rename_scene(project, 1, "Новая трактовка встречи")
print("\nПосле переименования:")
print(ask_project(project, "Как называется первая сцена?"))

Сцена «Встреча после разговора»:
Текущее название сцены: «Встреча после разговора».

После переименования:
Сцена «Новая трактовка встречи»:
Текущее название сцены: «Новая трактовка встречи».


## 22. Проверяем настоящий сценарий `save → edit → save → load → question`

Это важнее обычной проверки переменной в памяти. Мы действительно записываем проект на диск, загружаем его обратно и только потом задаём вопрос.

In [21]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "saved_project.json"

    save_project(project, path)
    loaded = load_project(path)

    rename_scene(loaded, 1, "Название после повторной загрузки")
    set_director_note(loaded, 1, "Правка после загрузки проекта")
    save_project(loaded, path)

    restored = load_project(path)
    print(ask_project(restored, "Как называется первая сцена?"))
    print(ask_project(restored, "Какая заметка режиссёра?"))

Сцена «Название после повторной загрузки»:
Текущее название сцены: «Название после повторной загрузки».
Сцена «Название после повторной загрузки»:
Заметка режиссёра: Правка после загрузки проекта


## 23. Автоматические проверки по критериям

In [22]:
# 1. Сцены существуют.
assert len(project["scenes"]) > 0

# 2. Персонаж из ремарки найден.
assert "Маша" in project["scenes"][0]["participants"]

# 3. У каждой сцены есть хотя бы один вариант решения.
assert all(scene["staging_options"] for scene in project["scenes"])

# 4. Во все постановочные решения попадают только персонажи из карточки сцены.
for scene in project["scenes"]:
    allowed = set(scene["participants"])
    for option in scene["staging_options"]:
        assert set(option["characters_used"]).issubset(allowed)

# 5. У фактических событий есть цитаты, когда событие строится по тексту.
for scene in project["scenes"]:
    for event in scene["events"]:
        assert event["type"] == "fact"
        assert event["quote"] in scene["source_text"] or event["quote"] == ""

# 6. Цели всегда помечены как интерпретации.
for scene in project["scenes"]:
    for goal in scene["goals"].values():
        assert goal["type"] == "interpretation"

# 7. Вариант решения обязан иметь опору на цитату.
for scene in project["scenes"]:
    assert all(option["based_on_quote"] for option in scene["staging_options"])

# 8. Сохранение действительно возвращает последнюю правку.
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "project.json"
    temp_project = deepcopy(project)
    rename_scene(temp_project, 1, "Проверка сохранения")
    save_project(temp_project, path)
    restored = load_project(path)
    assert "Проверка сохранения" in ask_project(restored, "Как называется первая сцена?")

# 9. Вопрос про объект, которого нет в проекте, не получает выдуманный ответ.
unknown = ask_project(project, "В этой пьесе происходит извержение вулкана?")
assert "не найдено" in unknown.lower() or "нет готового ответа" in unknown.lower()

print("Все проверки по критериям пройдены.")

Все проверки по критериям пройдены.


## 24. Что именно покрывает ноутбук

### Минимум

- ✅ художественный текст принимается из файла;
- ✅ текст делится на сцены;
- ✅ выделяются персонажи и их участие;
- ✅ создаётся карточка каждой сцены;
- ✅ факты подтверждаются цитатами;
- ✅ интерпретации отделены от фактов;
- ✅ отдельную сцену можно открыть;
- ✅ для каждой сцены есть несколько постановочных вариантов;
- ✅ вопросы работают по текущему сохранённому проекту.

### Сильные элементы

- ✅ линия персонажа через всю пьесу;
- ✅ литературный и пластический варианты;
- ✅ ограничения постановки;
- ✅ редактирование сцен, событий, целей и заметок;
- ✅ история изменений;
- ✅ сохранение и загрузка;
- ✅ защита от выдумывания отсутствующего материала;
- ✅ узкий и проверяемый конвейер без требования большой облачной модели.

Остаются предметные вопросы для кастдева с партнёром: какие ограничения важнее в реальной постановке, как именно режиссёр понимает участие героя и какой уровень генерации мизансцены нужен.

## 25. Опциональный интерфейс Streamlit

В ТЗ интерфейс `Streamlit` или `Gradio` назван достаточным вариантом. Сам ноутбук уже является рабочим прототипом и содержит всю логику.

Ниже не требуется запускать в этом ноутбуке. Это готовый простой каркас, который можно вынести в `app.py`, если для демонстрации нужен отдельный веб-интерфейс.

In [23]:
STREAMLIT_APP_TEMPLATE = r'''
import streamlit as st

st.title("ИИ-ассистент режиссёра")
st.write("Загрузите TXT-пьесу и получите структурированный проект.")

uploaded = st.file_uploader("Пьеса в формате TXT", type=["txt"])

if uploaded is not None:
    text = uploaded.read().decode("utf-8")
    st.session_state["play_text"] = text
    st.success("Текст загружен. Основной конвейер находится в Jupyter Notebook.")

st.info("В учебной версии основная реализация находится в Jupyter Notebook.")
'''

print(STREAMLIT_APP_TEMPLATE)


import streamlit as st

st.title("ИИ-ассистент режиссёра")
st.write("Загрузите TXT-пьесу и получите структурированный проект.")

uploaded = st.file_uploader("Пьеса в формате TXT", type=["txt"])

if uploaded is not None:
    text = uploaded.read().decode("utf-8")
    st.session_state["play_text"] = text
    st.success("Текст загружен. Основной конвейер находится в Jupyter Notebook.")

st.info("В учебной версии основная реализация находится в Jupyter Notebook.")



## 26. Как использовать ноутбук на настоящей пьесе

1. Сохраните текст пьесы в UTF-8 `.txt`.
2. Укажите путь в `TEXT_PATH`.
3. Выполните ячейки сверху вниз.
4. Просмотрите карточки сцен.
5. Проверьте персонажей, особенно тех, кто входит в ремарках.
6. При необходимости задайте ограничения постановки.
7. Измените трактовки и заметки режиссёра.
8. Сохраните проект через `save_project(...)`.
9. Загрузите проект через `load_project(...)` и задавайте вопросы уже ему.

Для проверки задания полезно брать другую пьесу из разрешённых источников, а не только демонстрационный текст.